# End-to-End Asset Embeddings Pipeline

This notebook walks through the project's canonical training chain — **W2V → BERT-PT → BERT-FT** — on synthetic data. The deliverable of each step is an asset-embedding CSV.

The configurations are tiny (sized for CPU), but the wiring is identical to production runs on real CSMAR data. End-to-end runtime is roughly one minute.

```text
   ┌─────────────────────────┐                     ┌─────────────────────────┐
   │ portfolio_pretrained.csv│                     │ portfolio_finetune.csv  │
   │      (PT period)        │                     │      (FT period)        │
   └────────────┬────────────┘                     └────────────┬────────────┘
                │                                               │
                ▼                                               │
       ┌─────────────────┐                                      │
       │  Step 1 · W2V   │                                      │
       │  d=16, 10 ep    │                                      │
       └────────┬────────┘                                      │
                │  init                                         │
                ▼                                               │
       ┌─────────────────┐                                      │
       │ Step 2 · BERT-PT│                                      │
       │ 2 layers, 8 ep  │                                      │
       └────────┬────────┘                                      │
                │  init                                         │
                ▼                                               │
       ┌─────────────────┐                                      │
       │ Step 3 · BERT-FT│ ◀────────────────────────────────────┘
       │   5 epochs      │
       └────────┬────────┘
                │
                ▼
        3 embedding CSVs (W2V, BERT-PT, BERT-FT)
```

**Reading the diagram.** BERT-PT inherits W2V's vocabulary and initial embedding matrix; BERT-FT warm-starts from BERT-PT and trains on a later-period portfolio set. These two initialization couplings keep the resulting embedding spaces in one coherent coordinate frame, so the fine-tuned embedding moves with the later period's holdings while staying comparable to its pretrained anchor. Details on the synthetic data design live in [`examples/data/README.md`](data/README.md); real CSMAR access is covered in [`docs/how-to/prepare-data.md`](../docs/how-to/prepare-data.md).

> **First time?** `jupyterlab` and `ipykernel` are demo-only extras and are **not** installed by a plain `uv sync`. Run once:
> ```bash
> uv sync --extra notebook
> uv run jupyter lab examples/quickstart.ipynb
> ```
> Prefer not to install Jupyter? Use the equivalent script: `uv run python examples/quickstart.py`.

In [1]:
import json
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

# Locate the project root by walking up to find pyproject.toml.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = PROJECT_ROOT / "examples" / "data"
WORK = PROJECT_ROOT / "examples" / "_output"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def run_step(cmd: list[str], step: str) -> None:
    t0 = time.time()
    res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=600)
    print(f"[{step}] returncode={res.returncode}  ({time.time()-t0:.1f}s)")
    if res.returncode != 0:
        print(res.stderr[-1500:])
        raise RuntimeError(f"{step} failed")
    return res


print("Project root:", PROJECT_ROOT)
print("Data dir   :", DATA)
print("Workdir    :", WORK)

Project root: D:\PythonProjects\AssetEmbeddings
Data dir   : D:\PythonProjects\AssetEmbeddings\examples\data
Workdir    : D:\PythonProjects\AssetEmbeddings\examples\_output


## Step 1 · Word2Vec on the pre-train period

Skip-gram on `portfolio_pretrained.csv`. Stocks that frequently co-occur in the same portfolios drift toward similar vectors — the "stocks as words" analogy. The output `.model` artifact doubles as BERT's tokenizer and initial embedding matrix in Step 2; the `*_embedding.csv` is the standalone Word2Vec embedding artifact.

| Setting | Value |
|---|---|
| Algorithm | skip-gram with negative sampling |
| Embedding dim | 16 |
| Epochs | 10 |
| Context window | 5 |

In [2]:
w2v_folder = WORK / "w2v"
w2v_name = "w2v_d16"

w2v_config = {
    "embedding_dim": 16,
    "epochs": 10,
    "window": 5,
    "min_count": 1,
    "sg": 1,
    "sample": 1e-3,
    "negative_sample": 3,
    "data_path": str(DATA / "portfolio_pretrained.csv"),
    "data_format": "csv",
    "workers": 1,
    "save_folder": str(w2v_folder),
    "save_name": w2v_name,
    "save_format": ".model",
    "seed": SEED,
}
w2v_config_path = WORK / "w2v_config.json"
write_json(w2v_config_path, w2v_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.w2v", "-c", str(w2v_config_path)], "W2V")

w2v_model_path = w2v_folder / f"{w2v_name}.model"
w2v_embedding_csv = w2v_folder / f"{w2v_name}_embedding.csv"

# BERT vocab = W2V vocab + 3 special tokens ([MASK], [PAD], [UNK]).
w2v = Word2Vec.load(str(w2v_model_path))
BERT_VOCAB = len(w2v.wv) + 3
print(f"W2V vocab: {len(w2v.wv)}  ->  BERT vocab: {BERT_VOCAB}")

[W2V] returncode=0  (8.6s)
W2V vocab: 2499  ->  BERT vocab: 2502


## Step 2 · BERT pretrain (MLM) on the pre-train period

Masked-language-model pretraining on the same PT portfolios. The W2V model from Step 1 supplies the vocabulary and warm-starts the embedding matrix; the transformer layers are randomly initialized.

| Setting | Value |
|---|---|
| Hidden dim | 16 (must match W2V) |
| Layers / heads | 2 / 2 |
| Intermediate size | 64 |
| Mask probability | 0.15 |
| Epochs | 8 |

Three CPU-friendly overrides vs. the production configs in `configs/pretrained/AssetBERT/`:

- `optimizer.optimizer_type: AdamW` — production uses `PagedAdam8bit`, which needs bitsandbytes + CUDA.
- `dataloader.num_workers: 0` — guards against Windows + Jupyter dataloader deadlocks.
- `train.mixed_precision: "no"` — explicit; AMP only helps on CUDA.

In [3]:
bert_pt_folder = WORK / "bert_pt"
bert_pt_name = "bert_pt_d16"

bert_pt_config = {
    "model": {
        "model_type": "BERT",
        "model_checkpoint": None,
        "w2v_model": str(w2v_model_path),
        "embedding_file": None,
        "vocab_size": BERT_VOCAB,
        "hidden_size": 16,
        "num_hidden_layers": 2,
        "num_attention_heads": 2,
        "intermediate_size": 64,
        "max_position_embeddings": 64,
        "type_vocab_size": 1,
        "freeze_embedding": False,
        "freeze_encoder": False,
    },
    "tokenizer": {
        "w2v_model": str(w2v_model_path),
        "vocab_file": None,
        "pretrained_tokenizer_file": None,
        "alias_file": None,
    },
    "dataset": {
        "data_path": str(DATA / "portfolio_pretrained.csv"),
        "data_format": "csv",
        "id_key": "InvestorID",
        "portfolio_key": "Portfolio",
        "proportion1_key": "Proportion1",
        "proportion2_key": "Proportion2",
        "include_proportion": False,
        "max_length": 64,
        "num_repeats": 1,
        "mask_prob": 0.15,
        "mask_indices": None,
        "cache_size": 100,
    },
    "dataloader": {
        "batch_size": 64,
        "shuffle": True,
        "num_workers": 0,
        "persistent_workers": False,
        "pin_memory": False,
        "drop_last": False,
    },
    "optimizer": {
        "optimizer_type": "AdamW",
        "optimizer_kwargs": None,
        "learning_rate": 0.001,
        "lr_scheduler_type": "cosine",
        "lr_scheduler_warmup_steps": 0,
        "lr_scheduler_train_steps": None,
        "lr_scheduler_num_cycles": 0.5,
        "lr_scheduler_power": 0,
    },
    "train": {
        "max_epoches": 8,
        "accelerator_checkpoint": None,
        "validation_split": 0.2,
        "split_method": "random",
        "best_metric": "val_loss",
        "clip_grad_norm": None,
        "clip_grad_value": None,
        "gradient_accumulation_steps": 1,
        "detect_anomaly": False,
        "mixed_precision": "no",
        "check_per_step": 200,
        "report_per_epoch": 1,
        "calculate_contextualized_embeddings": True,
        "save_per_epoch": 1,
        "save_folder": str(bert_pt_folder),
        "save_name": bert_pt_name,
        "save_format": ".safetensors",
        "seed": SEED,
    },
}
bert_pt_config_path = WORK / "bert_pt_config.json"
write_json(bert_pt_config_path, bert_pt_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.bert", "-c", str(bert_pt_config_path)], "BERT-PT")

bert_pt_checkpoint = bert_pt_folder / f"{bert_pt_name}_best" / "model.safetensors"
bert_pt_embedding_csv = bert_pt_folder / f"{bert_pt_name}_best_contextual_embedding.csv"
assert bert_pt_checkpoint.exists() and bert_pt_embedding_csv.exists()

[BERT-PT] returncode=0  (14.4s)


## Step 3 · BERT fine-tune on the FT period

Warm-start from the BERT-PT checkpoint and continue MLM training on `portfolio_finetune.csv`. The FT period differs from PT by a cluster-level drift in latent space (see [`data/README.md`](data/README.md)) — fine-tuning is what teaches the model that drift direction.

Five things change vs. the PT config:

- `model.model_checkpoint` — point at the PT `.safetensors`.
- `dataset.data_path` — FT-period portfolios.
- `dataset.include_proportion: true` — proportions enter as an auxiliary signal.
- `optimizer.learning_rate: 5e-4` — gentler than the 1e-3 used for PT.
- `train.max_epoches: 5` and `dataloader.batch_size: 32` — shorter, calmer schedule.

In [4]:
bert_ft_folder = WORK / "bert_ft"
bert_ft_name = "bert_ft_d16"

bert_ft_config = dict(bert_pt_config)
bert_ft_config["model"] = {**bert_pt_config["model"], "model_checkpoint": str(bert_pt_checkpoint)}
bert_ft_config["dataset"] = {
    **bert_pt_config["dataset"],
    "data_path": str(DATA / "portfolio_finetune.csv"),
    "include_proportion": True,
}
bert_ft_config["dataloader"] = {**bert_pt_config["dataloader"], "batch_size": 32}
bert_ft_config["optimizer"] = {**bert_pt_config["optimizer"], "learning_rate": 5e-4}
bert_ft_config["train"] = {
    **bert_pt_config["train"],
    "max_epoches": 5,
    "save_folder": str(bert_ft_folder),
    "save_name": bert_ft_name,
}

bert_ft_config_path = WORK / "bert_ft_config.json"
write_json(bert_ft_config_path, bert_ft_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.bert", "-c", str(bert_ft_config_path)], "BERT-FT")

bert_ft_embedding_csv = bert_ft_folder / f"{bert_ft_name}_best_contextual_embedding.csv"
assert bert_ft_embedding_csv.exists()

[BERT-FT] returncode=0  (15.3s)


## Results · the three embedding artifacts

Each training step wrote a `(Token, Embed_1..Embed_d)` CSV — the asset-embedding deliverable. We load all three, summarize their shapes, and peek at the nearest neighbours of one stock in each space. The synthetic universe has latent cluster structure, so a stock's neighbours should stay within its cluster.

In [ ]:
artifacts = {
    "W2V": (w2v_embedding_csv, "PT data"),
    "BERT-PT": (bert_pt_embedding_csv, "PT data"),
    "BERT-FT": (bert_ft_embedding_csv, "FT data"),
}

embeddings = {}
rows = []
for label, (csv_path, trained) in artifacts.items():
    df = pd.read_csv(csv_path, dtype={"Token": str})
    embeddings[label] = df.set_index("Token")
    rows.append(
        {
            "Trained on": trained,
            "Shape": f"{df.shape[0]} x {df.shape[1] - 1}",
            "Path": str(csv_path.relative_to(PROJECT_ROOT)),
        }
    )

# The token column holds special tokens like [CLS] alongside stock codes, so skip those when probing.
probe = next(t for t in embeddings["W2V"].index if not t.startswith("["))
for label, emb in embeddings.items():
    x = emb.to_numpy()
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    sims = x @ x[emb.index.get_loc(probe)]
    order = np.argsort(-sims)
    neighbours = [emb.index[i] for i in order if emb.index[i] != probe and not emb.index[i].startswith("[")][:5]
    rows[list(embeddings).index(label)][f"Top-5 neighbours of {probe}"] = ", ".join(neighbours)

summary = pd.DataFrame(rows, index=list(artifacts.keys()))
summary

## Interpretation

**W2V and BERT-PT recover the same structure** because they are both trained on the same PT-period portfolios: their neighbour lists reflect the PT-period cluster assignments.

**BERT-FT moves with the data.** Fine-tuning on FT-period portfolios lets it pick up the cluster-level drift between the two periods — while the W2V→BERT and PT→FT initialization couplings keep its coordinate frame anchored to the pretrained space. On real CSMAR data this coherence is what makes quarter-by-quarter embedding sequences comparable over time.

## Where to go next

- [`docs/how-to/train.md`](../docs/how-to/train.md) — full training pipeline on CSMAR data, all model variants
- [`docs/how-to/prepare-data.md`](../docs/how-to/prepare-data.md) — CSMAR data acquisition and processing
- [`docs/explanation/method.md`](../docs/explanation/method.md) — the portfolio-sentence method and the coupled pretrain–finetune design